# 03 — Exploratory Data Analysis (EDA)
**QM640 Data Analytics Capstone | Walsh College**

This notebook explores the cleaned Fragrantica dataset to:
- Understand rating distributions
- Examine note and accord frequency
- Compare ratings across concentration and gender groups
- Explore the relationship between popularity (num_votes) and rating

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_theme(style="whitegrid", palette="muted")
FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/processed/fragrantica_clean.csv")
print(f"Shape: {df.shape}")
df.head()

## 3.1 Rating Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["rating_score"].dropna(), bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Distribution of Rating Scores")
axes[0].set_xlabel("Rating Score (0–5)")
axes[0].set_ylabel("Count")

axes[1].hist(df["num_votes"].dropna(), bins=60, color="coral", edgecolor="white")
axes[1].set_title("Distribution of Vote Counts")
axes[1].set_xlabel("Number of Votes")
axes[1].set_ylabel("Count")
axes[1].set_yscale("log")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_rating_vote_distributions.png", dpi=150)
plt.show()

## 3.2 Ratings by Concentration (RQ2)

In [ ]:
conc_order = ["Parfum", "Eau De Parfum", "Eau De Toilette", "Eau De Cologne", "Body Mist"]
df_conc = df[df["concentration"].isin(conc_order)]

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_conc, x="concentration", y="rating_score", order=conc_order)
plt.title("Rating Score by Concentration Level (RQ2)")
plt.xlabel("Concentration")
plt.ylabel("Rating Score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(FIG_DIR / "02_rating_by_concentration.png", dpi=150)
plt.show()

## 3.3 Top 20 Most Frequent Notes

In [ ]:
def count_notes(series):
    all_notes = []
    for val in series.dropna():
        all_notes.extend([n.strip() for n in str(val).split("|") if n.strip()])
    return Counter(all_notes)

all_note_counts = count_notes(df["top_notes"]) + count_notes(df["heart_notes"]) + count_notes(df["base_notes"])
top20_notes = pd.DataFrame(all_note_counts.most_common(20), columns=["note", "count"])

plt.figure(figsize=(10, 6))
sns.barh(top20_notes["note"][::-1], top20_notes["count"][::-1], color="mediumseagreen")
plt.title("Top 20 Most Frequent Notes Across All Fragrances")
plt.xlabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_top20_notes.png", dpi=150)
plt.show()

## 3.4 Ratings by Gender Label (RQ3)

In [ ]:
plt.figure(figsize=(7, 5))
sns.violinplot(data=df, x="gender_label", y="rating_score",
               order=["Women", "Men", "Unisex"], inner="quartile")
plt.title("Rating Score by Gender Label (RQ3)")
plt.xlabel("Gender Label")
plt.ylabel("Rating Score")
plt.tight_layout()
plt.savefig(FIG_DIR / "04_rating_by_gender.png", dpi=150)
plt.show()

## 3.5 Popularity vs. Rating (RQ4)

In [ ]:
sample = df.sample(min(3000, len(df)), random_state=42)

plt.figure(figsize=(8, 5))
plt.scatter(np.log1p(sample["num_votes"]), sample["rating_score"],
            alpha=0.3, s=10, color="slateblue")
plt.title("log(Votes + 1) vs. Rating Score (RQ4)")
plt.xlabel("log(num_votes + 1)")
plt.ylabel("Rating Score")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_popularity_vs_rating.png", dpi=150)
plt.show()